In [1]:
import pdfplumber #library for extracting text from pdfs
import pandas as pd
import re #regular expressions for pattern matching in text

In [2]:
period1_files = {
    2007: 'appendixb_2007.pdf',
    2008: 'appendixb_2008.pdf'
}

In [3]:
states = {'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado',
    'Connecticut', 'Delaware', 'District of Columbia', 'Florida', 'Georgia',
    'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky',
    'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota',
    'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire',
    'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota',
    'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island',
    'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont',
    'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming'}

In [4]:
skip_regions = {'NORTHEAST REGION', 'NORTH CENTRAL REGION', 'SOUTH REGION', 'WEST REGION',
    'New England Division', 'Middle Atlantic Division',
    'East North Central Division', 'West North Central Division',
    'South Atlantic Division', 'East South Central Division',
    'West South Central Division', 'Mountain Division', 'Pacific Division',
    'UNITED STATES SUBTOTAL', 'TERRITORIES, POSSESSIONS,',
    'TERRITORIES, POSSESSIONS, OR', 'TOTAL', 'OR UNKNOWN'}

In [5]:
print(f'Tracking {len(states)} states')

Tracking 51 states


In [6]:
#clean number strings
#convert numerical strings like '1234' or '84.52%' to float. Returns None if string cannot be converted.
def to_float(s):
    if s is None:
        return None
    s = str(s).strip().replace(',', '').replace('%','') #remove commas adn percent signs
    try:
        return float(s)
    except (ValueError, TypeError):
        return None

# print(to_float('1,234'))
# print(to_float('85.34%'))
# print(to_float(''))
# print(to_float('Maine'))

In [7]:
def find_table_pages(pdf):
    table_pages = {}
    current_table = None
    
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ''
        lines = text.split('\n')
        header = ' '.join(lines[:3])
        
        for tname in ['B-46', 'B-47', 'B-48', 'B-49']:
            if f'Table {tname}' in header:
                current_table = tname.replace('-', '')
                if current_table not in table_pages:
                    table_pages[current_table] = []
                break
        
        if current_table:
            has_data = any(s in text for s in ['Maine', 'Ohio', 'Texas', 'Hawaii'])
            if has_data and i not in table_pages.get(current_table, []):
                table_pages.setdefault(current_table, []).append(i)
                
    
    return table_pages

In [8]:
#table B46 gender parser
def parse_gender_page(text):
    rows = []
    lines = text.split('\n')

    for line in lines:
        line = line.strip()

        if not line: #skip empty lines
            continue

        skip = False
        for skip_text in skip_regions:
            if line.startswith(skip_text):
                skip = True
                break
        if skip:
                continue
        if any(line.startswith(word) for word in
                   ['Table', 'Census', 'State', 'Source', 'MALES', 'MEN', 
                'Rows', 'DoD', '#', '%']):
            continue

             # Now try to match the data pattern
        # re.match only matches at the START of the string
        # Pattern explanation:
        #   ([A-Za-z][A-Za-z\s\.]+?)  = state name (lazy match, stops at first number)
        #   \s+                        = whitespace separator
        #   ([\d,]+)                   = males count (digits and commas)
        #   \s+[\d\.]+%?\s+           = males percentage (we capture but ignore)
        #   ([\d,]+)                   = females count
        #   \s+[\d\.]+%?\s+           = females percentage (ignore)
        #   ([\d,]+)                   = total count
        match = re.match(
            r'^([A-Za-z][A-Za-z\s\.]+?)\s+'   # group 1: state name
            r'([\d,]+)\s+'                      # group 2: dod males count
            r'([\d\.]+)%?\s+'                   # group 3: dod males pct
            r'([\d,]+)\s+'                      # group 4: dod females count
            r'([\d\.]+)%?\s+'                   # group 5: dod females pct
            r'([\d,]+)\s+'                      # group 6: dod total count
            r'([\d\.]+)%?\s+'                   # group 7: dod total pct
            r'([\d\.]+)%?\s+'                   # group 8: civ males pct
            r'([\d\.]+)%?\s+'                   # group 9: civ females pct
            r'([\d\.]+)%?',                     # group 10: civ total pct
            line
            )

        if match:
            state_name = match.group(1).strip()
            if state_name in states:
                    rows.append({
                        'state': state_name,
                        'dod_males': to_float(match.group(2)),
                        'dod_males_pct': to_float(match.group(3)),
                        'dod_females': to_float(match.group(4)),
                        'dod_females_pct': to_float(match.group(5)),
                        'dod_total': to_float(match.group(6)),
                        'dod_total_pct': to_float(match.group(7)),
                        'civ_males_pct': to_float(match.group(8)),
                        'civ_females_pct': to_float(match.group(9)),
                        'civ_total_pct': to_float(match.group(10)),
                        })
    return rows

#test


In [47]:
def parse_race_page(text):
    """
    Parse one page of the Period 1 B-47 race table.
    
    Three types of pages exist in B-47:
    - DoD race pages (3-5):      8 pairs (White,Black,AIAN,Asian,NHPI,Two+,Unknown,Total)
    - Civilian race pages (6-8): 7 pairs (White,Black,AIAN,Asian,NHPI,Two+,Total) no Unknown
    - Hispanic pages (9-11):     skip entirely
    
    Returns tuple: (dod_rows, civ_rows)
    """
    dod_rows = []
    civ_rows = []
    
    # ── Determine which type of page this is ────────────────────────────────
    # Look at first few lines for identifying keywords
    lines = text.split('\n')
    header_block = ' '.join(lines[:6]).upper()
    
    # Hispanic pages have HISPANIC and NOT HISPANIC in header
    # but do NOT have WHITE in the column headers
    is_hispanic = (
        'HISPANIC' in header_block and
        'NOT HISPANIC' in header_block
    )
    if is_hispanic:
        return [], []  # skip entirely
    
    # Civilian pages have '18-24 YEAR-OLD CIVILIANS' in header
    # but do NOT have 'DoD' before it
    # is_civilian = ((
    #     '18-24 YEAR-OLD CIVILIANS' in header_block or
    #     'CIVILIAN POPULATION, 18-24 YEARS-OLD' in header_block or
    #     'CIVILIAN POPULATION 18' in header_block) and 
    #     'DOD' not in header_block
    #               )   

    is_civilian = (
            'CIVILIAN' in header_block and
            '18' in header_block and
            'DOD' not in header_block and
            'HISPANIC' not in header_block
                    )
    
    
    # DoD pages have 'DoD' in header or just WHITE BLACK columns
    # with no civilian mention
    is_dod = not is_hispanic and not is_civilian
    
    # ── Debug: confirm page type ─────────────────────────────────────────────
    page_type = 'HISPANIC' if is_hispanic else ('CIVILIAN' if is_civilian else 'DOD')
    # Uncomment next line to debug:
    # print(f'  Page type detected: {page_type}')
    
    # ── Parse lines ──────────────────────────────────────────────────────────
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        # Skip headers, regions, divisions, totals, footnotes
        skip = False
        for skip_text in skip_regions:
            if line.startswith(skip_text):
                skip = True
                break
        if skip:
            continue
        
        if any(line.startswith(word) for word in
               ['Table', 'Census', 'State', 'Source', 'WHITE', 'Rows',
                'DoD', '#', '%', 'AIAN', 'Note', '1.', '2.', 'Page']):
            continue
        
        if is_dod:
            # 8 pairs: White, Black, AIAN, Asian, NHPI, Two+, Unknown, Total
            match = re.match(
                r'^([A-Za-z][A-Za-z\s\.]+?)\s+'   # state name
                r'([\d,]+)\s+([\d\.]+)%?\s+'        # white count + pct
                r'([\d,]+)\s+([\d\.]+)%?\s+'        # black count + pct
                r'([\d,]+)\s+([\d\.]+)%?\s+'        # aian count + pct
                r'([\d,]+)\s+([\d\.]+)%?\s+'        # asian count + pct
                r'([\d,]+)\s+([\d\.]+)%?\s+'        # nhpi count + pct
                r'([\d,]+)\s+([\d\.]+)%?\s+'        # two or more count + pct
                r'([\d,]+)\s+([\d\.]+)%?\s+'        # unknown count + pct
                r'([\d,]+)\s+([\d\.]+)%?',           # total count + pct
                line
            )
            if match:
                state_name = match.group(1).strip()
                if state_name in states:
                    dod_rows.append({
                        'state':            state_name,
                        'dod_white':        to_float(match.group(2)),
                        'dod_white_pct':    to_float(match.group(3)),
                        'dod_black':        to_float(match.group(4)),
                        'dod_black_pct':    to_float(match.group(5)),
                        'dod_aian':         to_float(match.group(6)),
                        'dod_aian_pct':     to_float(match.group(7)),
                        'dod_asian':        to_float(match.group(8)),
                        'dod_asian_pct':    to_float(match.group(9)),
                        'dod_nhpi':         to_float(match.group(10)),
                        'dod_nhpi_pct':     to_float(match.group(11)),
                        'dod_two_or_more':  to_float(match.group(12)),
                        'dod_two_pct':      to_float(match.group(13)),
                        'dod_unknown':      to_float(match.group(14)),
                        'dod_unknown_pct':  to_float(match.group(15)),
                        'dod_total_race':   to_float(match.group(16)),
                        'dod_total_pct':    to_float(match.group(17)),
                    })
        
        elif is_civilian:
            # 7 pairs: White, Black, AIAN, Asian, NHPI, Two+, Total
            # No Unknown column in civilian section
            match = re.match(
                r'^([A-Za-z][A-Za-z\s\.]+?)\s+'   # state name
                r'([\d,]+)\s+([\d\.]+)%?\s+'        # white count + pct
                r'([\d,]+)\s+([\d\.]+)%?\s+'        # black count + pct
                r'([\d,]+)\s+([\d\.]+)%?\s+'        # aian count + pct
                r'([\d,]+)\s+([\d\.]+)%?\s+'        # asian count + pct
                r'([\d,]+)\s+([\d\.]+)%?\s+'        # nhpi count + pct
                r'([\d,]+)\s+([\d\.]+)%?\s+'        # two or more count + pct
                r'([\d,]+)\s+([\d\.]+)%?',           # total count + pct
                line
            )
            if match:
                state_name = match.group(1).strip()
                if state_name in states:
                    civ_rows.append({
                        'state':            state_name,
                        'civ_white':        to_float(match.group(2)),
                        'civ_white_pct':    to_float(match.group(3)),
                        'civ_black':        to_float(match.group(4)),
                        'civ_black_pct':    to_float(match.group(5)),
                        'civ_aian':         to_float(match.group(6)),
                        'civ_aian_pct':     to_float(match.group(7)),
                        'civ_asian':        to_float(match.group(8)),
                        'civ_asian_pct':    to_float(match.group(9)),
                        'civ_nhpi':         to_float(match.group(10)),
                        'civ_nhpi_pct':     to_float(match.group(11)),
                        'civ_two_or_more':  to_float(match.group(12)),
                        'civ_two_pct':      to_float(match.group(13)),
                        'civ_total':        to_float(match.group(14)),
                        'civ_total_pct':    to_float(match.group(15)),
                    })
    
    return dod_rows, civ_rows

In [10]:
#store results for each year
both_years_gender = []
both_years_dod_race = []
both_years_civ_race = []

for year, pdf_path in period1_files.items():
    print(f'\n=== Processing {year} ===')

    gender_rows = []
    dod_race_rows = []
    civ_race_rows = []
    
    with pdfplumber.open(pdf_path) as pdf:
        tp = find_table_pages(pdf)
        print(f'B46 pages: {tp.get("B46")}')
        print(f'B47 pages: {tp.get("B46")}')

        #gender table
        #gender_rows = []
        for page_idx in tp.get('B46', []):
            text = pdf.pages[page_idx].extract_text()
            rows = parse_gender_page(text)
            gender_rows.extend(rows)

        df_gender = pd.DataFrame(gender_rows).drop_duplicates(subset='state')
        df_gender['year'] = year
        print(f'Gender rows: {len(df_gender)}')

        #race table
        #dod_race_rows = []
        #civ_race_rows = []
        for page_idx in tp.get('B47', []):
            text = pdf.pages[page_idx].extract_text()
            dod_rows, civ_rows = parse_race_page(text)
            dod_race_rows.extend(dod_rows)
            civ_race_rows.extend(civ_rows)

        df_dod_race = pd.DataFrame(dod_race_rows).drop_duplicates(subset='state')
        df_civ_race = pd.DataFrame(civ_race_rows).drop_duplicates(subset='state')
        df_dod_race['year'] = year
        df_civ_race['year'] = year
        print(f'DoD race rows: {len(df_dod_race)}')
        print(f'Civilian race rows: {len(df_civ_race)}')

    both_years_gender.append(df_gender)
    both_years_dod_race.append(df_dod_race)
    both_years_civ_race.append(df_civ_race)

#stack all years into single data frames
df_gender_p1 = pd.concat(both_years_gender, ignore_index=True)
df_dod_race_p1 = pd.concat(both_years_dod_race, ignore_index=True)
df_civ_race_p1 = pd.concat(both_years_civ_race, ignore_index=True)

print(f'\n=== Period 1 Summary ===')
print(f'Gender rows: {len(df_gender_p1)} (expected {51 * len(period1_files)})')
print(f'DoD race rows: {len(df_dod_race_p1)} (expected {51 * len(period1_files)})')
print(f'Civilian race rows: {len(df_civ_race_p1)} (expected {51 * len(period1_files)})')
print(f'Years covered: {sorted(df_gender_p1.year.unique())}')


=== Processing 2007 ===
B46 pages: [0, 1, 2]
B47 pages: [0, 1, 2]
Gender rows: 51
DoD race rows: 51
Civilian race rows: 51

=== Processing 2008 ===
B46 pages: [0, 1, 2]
B47 pages: [0, 1, 2]
Gender rows: 51
DoD race rows: 51
Civilian race rows: 51

=== Period 1 Summary ===
Gender rows: 102 (expected 102)
DoD race rows: 102 (expected 102)
Civilian race rows: 102 (expected 102)
Years covered: [2007, 2008]


In [11]:
print(df_civ_race_p1.head(5))

           state  civ_white  civ_white_pct  civ_black  civ_black_pct  \
0          Maine   118869.0          95.57     1513.0           1.22   
1  New Hampshire   117276.0          95.86     1932.0           1.58   
2        Vermont    58548.0          95.12      512.0           0.83   
3  Massachusetts   496359.0          87.13    38442.0           6.75   
4   Rhode Island    87166.0          88.21     7131.0           7.22   

   civ_aian  civ_aian_pct  civ_asian  civ_asian_pct  civ_nhpi  civ_nhpi_pct  \
0     382.0          0.31      786.0           0.63       0.0          0.00   
1       0.0          0.00     1903.0           1.56     106.0          0.09   
2     333.0          0.54      879.0           1.43       0.0          0.00   
3    1777.0          0.31    28253.0           4.96       0.0          0.00   
4      89.0          0.09     1972.0           2.00      42.0          0.04   

   civ_two_or_more  civ_two_pct  civ_total  civ_total_pct  year  
0           2829.0        

In [12]:
# Check that 2007 and 2008 numbers are actually different
# Pick a few states and compare totals across years

states_to_check = ['Tennessee', 'Texas', 'California', 'Maine']

print('=== Cross-year validation ===')
print()

for state in states_to_check:
    rows = df_gender_p1[df_gender_p1.state == state][['year', 'state', 'dod_total', 'dod_males', 'dod_females']]
    print(rows.to_string(index=False))
    print()

print('=== Race cross-check ===')
for state in ['Tennessee', 'Texas']:
    rows = df_dod_race_p1[df_dod_race_p1.state == state][['year', 'state', 'dod_white', 'dod_black', 'dod_total_race']]
    print(rows.to_string(index=False))
    print()

print('=== Civilian population cross-check ===')
for state in ['Tennessee', 'Texas']:
    rows = df_civ_race_p1[df_civ_race_p1.state == state][['year', 'state', 'civ_white', 'civ_black', 'civ_total']]
    print(rows.to_string(index=False))
    print()

=== Cross-year validation ===

 year     state  dod_total  dod_males  dod_females
 2007 Tennessee     3375.0     2896.0        479.0
 2008 Tennessee     3493.0     3045.0        448.0

 year state  dod_total  dod_males  dod_females
 2007 Texas    17215.0    14255.0       2960.0
 2008 Texas    18021.0    14934.0       3087.0

 year      state  dod_total  dod_males  dod_females
 2007 California    16429.0    13682.0       2747.0
 2008 California    17949.0    15135.0       2814.0

 year state  dod_total  dod_males  dod_females
 2007 Maine      846.0      715.0        131.0
 2008 Maine      851.0      747.0        104.0

=== Race cross-check ===
 year     state  dod_white  dod_black  dod_total_race
 2007 Tennessee     2770.0      487.0          3375.0
 2008 Tennessee     2747.0      573.0          3493.0

 year state  dod_white  dod_black  dod_total_race
 2007 Texas    13425.0     2157.0         17215.0
 2008 Texas    13340.0     2348.0         18021.0

=== Civilian population cross-check

In [13]:
#merge gender, race
df_2007_2008 = pd.merge(df_gender_p1, df_dod_race_p1, on=['state', 'year'], how='outer')
df_2007_2008 = pd.merge(df_2007_2008, df_civ_race_p1, on=['state', 'year'], how='outer')

In [14]:
#civilian count
df_2007_2008['civ_males'] = df_2007_2008['civ_total'] * df_2007_2008['civ_males_pct']/100
df_2007_2008['civ_females'] = df_2007_2008['civ_total'] * df_2007_2008['civ_females_pct']/100

In [15]:
print(f'Period 1 panel shape: {df_2007_2008.shape}')
print(f'Columns: {list(df_2007_2008.columns)}')
print()
print(df_2007_2008.head())

Period 1 panel shape: (102, 43)
Columns: ['state', 'dod_males', 'dod_males_pct', 'dod_females', 'dod_females_pct', 'dod_total', 'dod_total_pct_x', 'civ_males_pct', 'civ_females_pct', 'civ_total_pct_x', 'year', 'dod_white', 'dod_white_pct', 'dod_black', 'dod_black_pct', 'dod_aian', 'dod_aian_pct', 'dod_asian', 'dod_asian_pct', 'dod_nhpi', 'dod_nhpi_pct', 'dod_two_or_more', 'dod_two_pct', 'dod_unknown', 'dod_unknown_pct', 'dod_total_race', 'dod_total_pct_y', 'civ_white', 'civ_white_pct', 'civ_black', 'civ_black_pct', 'civ_aian', 'civ_aian_pct', 'civ_asian', 'civ_asian_pct', 'civ_nhpi', 'civ_nhpi_pct', 'civ_two_or_more', 'civ_two_pct', 'civ_total', 'civ_total_pct_y', 'civ_males', 'civ_females']

           state  dod_males  dod_males_pct  dod_females  dod_females_pct  \
0          Maine      715.0          84.52        131.0            15.48   
1  New Hampshire      603.0          86.51         94.0            13.49   
2        Vermont      212.0          88.33         28.0            11.

In [16]:
PDF_2009 = 'appendixb_2009.pdf'

with pdfplumber.open(PDF_2009) as pdf:
    print(f'Total pages: {len(pdf.pages)}')
    print()
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ''
        first_line = text.split('\n')[0] if text else 'EMPTY'
        print(f'Page {i}: {repr(first_line[:80])}')

Total pages: 15

Page 0: 'Table B-46. FY 2009 Non-Prior Service (NPS) Active Component Enlisted Accessions'
Page 1: 'Table B-46. FY 2009 Non-Prior Service (NPS) Active Component Enlisted Accessions'
Page 2: 'Table B-46. FY 2009 Non-Prior Service (NPS) Active Component Enlisted Accessions'
Page 3: 'Table B-47. FY 2009 Non-Prior Service (NPS) Active Component Enlisted Accessions'
Page 4: 'Table B-47. FY 2009 Non-Prior Service (NPS) Active Component Enlisted Accessions'
Page 5: 'Table B-47. FY 2009 Non-Prior Service (NPS) Active Component Enlisted Accessions'
Page 6: 'Table B-48. FY 2009 Non-Prior Service (NPS) Active Component Enlisted Accessions'
Page 7: 'Table B-48. FY 2009 Non-Prior Service (NPS) Active Component Enlisted Accessions'
Page 8: 'Table B-48. FY 2009 Non-Prior Service (NPS) Active Component Enlisted Accessions'
Page 9: 'Table B-48. FY 2009 NPS Active Component Enlisted Accessions by Census Region, D'
Page 10: 'Table B-48. FY 2009 NPS Active Component Enlisted Accessions by

In [17]:
with pdfplumber.open('appendixb_2009.pdf') as pdf:
    for page_idx in [6, 9, 12]:
        text = pdf.pages[page_idx].extract_text()
        lines = text.split('\n')
        print(f'--- Page {page_idx} first 4 lines ---')
        for line in lines[:4]:
            print(repr(line))
        print()

--- Page 6 first 4 lines ---
'Table B-48. FY 2009 Non-Prior Service (NPS) Active Component Enlisted Accessions by Census Region, Division, State, and Race/Ethnicity with Civilian Comparison Group'
'CENSUS REGION DoD'
'Census Division WHITE BLACK AIAN ASIAN NHPI TWO OR MORE UNKNOWN TOTAL'
'State # % # % # % # % # % # % # % # %'

--- Page 9 first 4 lines ---
'Table B-48. FY 2009 NPS Active Component Enlisted Accessions by Census Region, Division, State, and Race/Ethnicity with Civilian Comparison Group'
'CENSUS REGION 18-24 YEAR-OLD CIVILIANS'
'Census Division WHITE BLACK AIAN ASIAN NHPI TWO OR MORE TOTAL'
'State # % # % # % # % # % # % # %'

--- Page 12 first 4 lines ---
'Table B-48. FY 2009 NPS Active Component Enlisted Accessions by Census Region, Division, State, and Race/Ethnicity with Civilian Comparison Group'
'CENSUS REGION DoD 18-24 YEAR-OLD CIVILIANS'
'Census Division HISPANIC NOT HISPANIC1 TOTAL HISPANIC NOT HISPANIC TOTAL'
'State # % # % # % # % # % # %'



In [18]:
period2_files = {
    2009: 'appendixb_2009.pdf',
    2010: 'appendixb_2010.pdf',
    2011: 'appendixb_2011.pdf',
    2012: 'appendixb_2012.pdf',
    2013: 'appendixb_2013.pdf',
    2014: 'appendixb_2014.pdf'
}

In [25]:
p2_years_gender = []
p2_years_dod_race = []
p2_years_civ_race = []


for year, pdf_path in period2_files.items():
    gender_rows = []
    dod_race_rows = []
    civ_race_rows = []
    
    with pdfplumber.open(pdf_path) as pdf:
        tp = find_table_pages(pdf)
        
        # Period 2: gender is B47, race is B48
        # Skip B46 entirely by just not reading it
        for page_idx in tp.get('B47', []):
            text = pdf.pages[page_idx].extract_text()
            rows = parse_gender_page(text)
            gender_rows.extend(rows)
        
        for page_idx in tp.get('B48', []):
            text = pdf.pages[page_idx].extract_text()
            dod_rows, civ_rows = parse_race_page(text)
            dod_race_rows.extend(dod_rows)
            civ_race_rows.extend(civ_rows)
    
        df_gender = pd.DataFrame(gender_rows).drop_duplicates(subset='state')
        df_gender['year'] = year
            
        df_dod_race = pd.DataFrame(dod_race_rows).drop_duplicates(subset='state')
        df_civ_race = pd.DataFrame(civ_race_rows).drop_duplicates(subset='state')
        df_dod_race['year'] = year
        df_civ_race['year'] = year
        print(f'DoD race rows: {len(df_dod_race)}')
        print(f'Civilian race rows: {len(df_civ_race)}')

    p2_years_gender.append(df_gender)
    p2_years_dod_race.append(df_dod_race)
    p2_years_civ_race.append(df_civ_race)

DoD race rows: 51
Civilian race rows: 51
DoD race rows: 51
Civilian race rows: 51
DoD race rows: 51
Civilian race rows: 51
DoD race rows: 51
Civilian race rows: 51
DoD race rows: 51
Civilian race rows: 51
DoD race rows: 51
Civilian race rows: 51


In [26]:
#stack all years
df_gender_p2 = pd.concat(p2_years_gender, ignore_index=True)
df_dod_race_p2 = pd.concat(p2_years_dod_race, ignore_index=True)
df_civ_race_p2 = pd.concat(p2_years_civ_race, ignore_index=True)

In [27]:
# Check 1: correct row counts
print('=== Row counts ===')
print(f'Gender rows: {len(df_gender_p2)} (expected {51 * len(period2_files)})')
print(f'DoD race rows: {len(df_dod_race_p2)} (expected {51 * len(period2_files)})')
print(f'Civilian race rows: {len(df_civ_race_p2)} (expected {51 * len(period2_files)})')
print(f'Years covered: {sorted(df_gender_p2.year.unique())}')
print()

# Check 2: no duplicate state-year combinations
dupes_gender = df_gender_p2[df_gender_p2.duplicated(subset=['state','year'])]
dupes_race = df_dod_race_p2[df_dod_race_p2.duplicated(subset=['state','year'])]
print(f'Gender duplicates: {len(dupes_gender)}')
print(f'Race duplicates: {len(dupes_race)}')
print()

# Check 3: spot check known values from the PDFs
# From 2011 PDF - Tennessee total should be 3371
# From 2011 PDF - Florida total should be 11330
checks = [
    (2011, 'Tennessee', 3371),
    (2011, 'Florida',   11330),
    (2011, 'Texas',     15151),
    (2009, 'California', None),  # just print whatever we get
]

print('=== Spot checks ===')
for year, state, expected in checks:
    row = df_gender_p2[
        (df_gender_p2.year == year) & 
        (df_gender_p2.state == state)
    ]
    if len(row) == 0:
        print(f'{year} {state}: NOT FOUND')
        continue
    actual = row['dod_total'].values[0]
    if expected:
        status = 'OK' if actual == expected else f'MISMATCH (expected {expected})'
        print(f'{year} {state}: {actual} {status}')
    else:
        print(f'{year} {state}: {actual}')
print()

# Check 4: verify all 51 states present for each year
print('=== States per year ===')
for year in sorted(df_gender_p2.year.unique()):
    n = len(df_gender_p2[df_gender_p2.year == year])
    status = 'OK' if n == 51 else f'WARNING - only {n}'
    print(f'{year}: {n} states {status}')

=== Row counts ===
Gender rows: 306 (expected 306)
DoD race rows: 306 (expected 306)
Civilian race rows: 306 (expected 306)
Years covered: [2009, 2010, 2011, 2012, 2013, 2014]

Gender duplicates: 0
Race duplicates: 0

=== Spot checks ===
2011 Tennessee: 3371.0 OK
2011 Florida: 11330.0 OK
2011 Texas: 15151.0 OK
2009 California: 16725.0

=== States per year ===
2009: 51 states OK
2010: 51 states OK
2011: 51 states OK
2012: 51 states OK
2013: 51 states OK
2014: 51 states OK


In [28]:
print('=== States per year ===')
for year in sorted(df_gender_p2.year.unique()):
    n_gender = len(df_gender_p2[df_gender_p2.year == year])
    n_dod = len(df_dod_race_p2[df_dod_race_p2.year == year])
    n_civ = len(df_civ_race_p2[df_civ_race_p2.year == year])
    status = 'OK' if n_gender == 51 and n_dod == 51 and n_civ == 51 else 'WARNING'
    print(f'{year}: gender={n_gender}, dod_race={n_dod}, civ_race={n_civ} {status}')

=== States per year ===
2009: gender=51, dod_race=51, civ_race=51 OK
2010: gender=51, dod_race=51, civ_race=51 OK
2011: gender=51, dod_race=51, civ_race=51 OK
2012: gender=51, dod_race=51, civ_race=51 OK
2013: gender=51, dod_race=51, civ_race=51 OK
2014: gender=51, dod_race=51, civ_race=51 OK


In [29]:
df_gender_p2.shape

(306, 11)

In [30]:
df_gender_p2['year'].unique()

array([2009, 2010, 2011, 2012, 2013, 2014])

In [31]:
#merge gender, race
df_p2 = pd.merge(df_gender_p2, df_dod_race_p2, on=['state', 'year'], how='outer')
df_p2 = pd.merge(df_p2, df_civ_race_p2, on=['state', 'year'], how='outer')

In [32]:
#civilian count
df_p2['civ_males'] = df_p2['civ_total'] * df_p2['civ_males_pct']/100
df_p2['civ_females'] = df_p2['civ_total'] * df_p2['civ_females_pct']/100

In [33]:
print(f'Period 1 panel shape: {df_p2.shape}')
print(f'Columns: {list(df_p2.columns)}')
print()
print(df_p2.head())

Period 1 panel shape: (306, 43)
Columns: ['state', 'dod_males', 'dod_males_pct', 'dod_females', 'dod_females_pct', 'dod_total', 'dod_total_pct_x', 'civ_males_pct', 'civ_females_pct', 'civ_total_pct_x', 'year', 'dod_white', 'dod_white_pct', 'dod_black', 'dod_black_pct', 'dod_aian', 'dod_aian_pct', 'dod_asian', 'dod_asian_pct', 'dod_nhpi', 'dod_nhpi_pct', 'dod_two_or_more', 'dod_two_pct', 'dod_unknown', 'dod_unknown_pct', 'dod_total_race', 'dod_total_pct_y', 'civ_white', 'civ_white_pct', 'civ_black', 'civ_black_pct', 'civ_aian', 'civ_aian_pct', 'civ_asian', 'civ_asian_pct', 'civ_nhpi', 'civ_nhpi_pct', 'civ_two_or_more', 'civ_two_pct', 'civ_total', 'civ_total_pct_y', 'civ_males', 'civ_females']

           state  dod_males  dod_males_pct  dod_females  dod_females_pct  \
0          Maine      694.0           0.53        103.0             0.40   
1  New Hampshire      586.0           0.45         81.0             0.31   
2        Vermont      181.0           0.14         23.0             0.

In [35]:
#period 3
period3_files = {
    2015: 'appendixb_2015.pdf',
    2016: 'appendixb_2016.pdf',
    2017: 'appendixb_2017.pdf',
    2018: 'appendixb_2018.pdf',
    2019: 'appendixb_2019.pdf'
}

In [36]:
p3_years_gender = []
p3_years_dod_race = []
p3_years_civ_race = []


for year, pdf_path in period3_files.items():
    gender_rows = []
    dod_race_rows = []
    civ_race_rows = []
    
    with pdfplumber.open(pdf_path) as pdf:
        tp = find_table_pages(pdf)
        
        # Period 2: gender is B47, race is B48
        # Skip B46 entirely by just not reading it
        for page_idx in tp.get('B47', []):
            text = pdf.pages[page_idx].extract_text()
            rows = parse_gender_page(text)
            gender_rows.extend(rows)
        
        for page_idx in tp.get('B48', []):
            text = pdf.pages[page_idx].extract_text()
            dod_rows, civ_rows = parse_race_page(text)
            dod_race_rows.extend(dod_rows)
            civ_race_rows.extend(civ_rows)
    
        df_gender = pd.DataFrame(gender_rows).drop_duplicates(subset='state')
        df_gender['year'] = year
            
        df_dod_race = pd.DataFrame(dod_race_rows).drop_duplicates(subset='state')
        df_civ_race = pd.DataFrame(civ_race_rows).drop_duplicates(subset='state')
        df_dod_race['year'] = year
        df_civ_race['year'] = year
        print(f'DoD race rows: {len(df_dod_race)}')
        print(f'Civilian race rows: {len(df_civ_race)}')

    p3_years_gender.append(df_gender)
    p3_years_dod_race.append(df_dod_race)
    p3_years_civ_race.append(df_civ_race)

DoD race rows: 51
Civilian race rows: 51
DoD race rows: 51
Civilian race rows: 51
DoD race rows: 51
Civilian race rows: 51
DoD race rows: 51
Civilian race rows: 51
DoD race rows: 51
Civilian race rows: 51


In [37]:
#stack all years
df_gender_p3 = pd.concat(p3_years_gender, ignore_index=True)
df_dod_race_p3 = pd.concat(p3_years_dod_race, ignore_index=True)
df_civ_race_p3 = pd.concat(p3_years_civ_race, ignore_index=True)

In [38]:
# Check 1: correct row counts
print('=== Row counts ===')
print(f'Gender rows: {len(df_gender_p3)} (expected {51 * len(period3_files)})')
print(f'DoD race rows: {len(df_dod_race_p3)} (expected {51 * len(period3_files)})')
print(f'Civilian race rows: {len(df_civ_race_p3)} (expected {51 * len(period3_files)})')
print(f'Years covered: {sorted(df_gender_p3.year.unique())}')
print()

# Check 2: no duplicate state-year combinations
dupes_gender_p3 = df_gender_p3[df_gender_p3.duplicated(subset=['state','year'])]
dupes_race_p3 = df_dod_race_p3[df_dod_race_p3.duplicated(subset=['state','year'])]
print(f'Gender duplicates: {len(dupes_gender)}')
print(f'Race duplicates: {len(dupes_race)}')
print()

# Check 3: spot check known values from the PDFs
checks = [
    (2017, 'Tennessee', 3401),
    (2017, 'Florida',   12396),
    (2017, 'Texas',     17811),
    (2017, 'California', None),  # just print whatever we get
]

print('=== Spot checks ===')
for year, state, expected in checks:
    row = df_gender_p3[
        (df_gender_p3.year == year) & 
        (df_gender_p3.state == state)
    ]
    if len(row) == 0:
        print(f'{year} {state}: NOT FOUND')
        continue
    actual = row['dod_total'].values[0]
    if expected:
        status = 'OK' if actual == expected else f'MISMATCH (expected {expected})'
        print(f'{year} {state}: {actual} {status}')
    else:
        print(f'{year} {state}: {actual}')
print()

# Check 4: verify all 51 states present for each year
print('=== States per year ===')
for year in sorted(df_gender_p3.year.unique()):
    n = len(df_gender_p3[df_gender_p3.year == year])
    status = 'OK' if n == 51 else f'WARNING - only {n}'
    print(f'{year}: {n} states {status}')

=== Row counts ===
Gender rows: 255 (expected 255)
DoD race rows: 255 (expected 255)
Civilian race rows: 255 (expected 255)
Years covered: [2015, 2016, 2017, 2018, 2019]

Gender duplicates: 0
Race duplicates: 0

=== Spot checks ===
2017 Tennessee: 3401.0 OK
2017 Florida: 12396.0 OK
2017 Texas: 17811.0 OK
2017 California: 19698.0

=== States per year ===
2015: 51 states OK
2016: 51 states OK
2017: 51 states OK
2018: 51 states OK
2019: 51 states OK


In [39]:
print('=== States per year ===')
for year in sorted(df_gender_p3.year.unique()):
    n_gender = len(df_gender_p3[df_gender_p3.year == year])
    n_dod = len(df_dod_race_p3[df_dod_race_p3.year == year])
    n_civ = len(df_civ_race_p3[df_civ_race_p3.year == year])
    status = 'OK' if n_gender == 51 and n_dod == 51 and n_civ == 51 else 'WARNING'
    print(f'{year}: gender={n_gender}, dod_race={n_dod}, civ_race={n_civ} {status}')

=== States per year ===
2015: gender=51, dod_race=51, civ_race=51 OK
2016: gender=51, dod_race=51, civ_race=51 OK
2017: gender=51, dod_race=51, civ_race=51 OK
2018: gender=51, dod_race=51, civ_race=51 OK
2019: gender=51, dod_race=51, civ_race=51 OK


In [40]:
#merge gender, race
df_p3 = pd.merge(df_gender_p3, df_dod_race_p3, on=['state', 'year'], how='outer')
df_p3 = pd.merge(df_p3, df_civ_race_p3, on=['state', 'year'], how='outer')

In [41]:
#civilian count
df_p3['civ_males'] = df_p3['civ_total'] * df_p3['civ_males_pct']/100
df_p3['civ_females'] = df_p3['civ_total'] * df_p3['civ_females_pct']/100

In [43]:
print(f'Period 1 panel shape: {df_p3.shape}')
print(f'Columns: {list(df_p3.columns)}')

Period 1 panel shape: (255, 43)
Columns: ['state', 'dod_males', 'dod_males_pct', 'dod_females', 'dod_females_pct', 'dod_total', 'dod_total_pct_x', 'civ_males_pct', 'civ_females_pct', 'civ_total_pct_x', 'year', 'dod_white', 'dod_white_pct', 'dod_black', 'dod_black_pct', 'dod_aian', 'dod_aian_pct', 'dod_asian', 'dod_asian_pct', 'dod_nhpi', 'dod_nhpi_pct', 'dod_two_or_more', 'dod_two_pct', 'dod_unknown', 'dod_unknown_pct', 'dod_total_race', 'dod_total_pct_y', 'civ_white', 'civ_white_pct', 'civ_black', 'civ_black_pct', 'civ_aian', 'civ_aian_pct', 'civ_asian', 'civ_asian_pct', 'civ_nhpi', 'civ_nhpi_pct', 'civ_two_or_more', 'civ_two_pct', 'civ_total', 'civ_total_pct_y', 'civ_males', 'civ_females']


In [44]:
#period 4
period4_files = {
    2020: 'appendixb_2020.pdf',
    2021: 'appendixb_2021.pdf',
    2022: 'appendixb_2022.pdf'
}

In [48]:
p4_years_gender = []
p4_years_dod_race = []
p4_years_civ_race = []


for year, pdf_path in period4_files.items():
    gender_rows = []
    dod_race_rows = []
    civ_race_rows = []
    
    with pdfplumber.open(pdf_path) as pdf:
        tp = find_table_pages(pdf)
        
        # Period 2: gender is B47, race is B48
        # Skip B46 entirely by just not reading it
        for page_idx in tp.get('B47', []):
            text = pdf.pages[page_idx].extract_text()
            rows = parse_gender_page(text)
            gender_rows.extend(rows)
        
        for page_idx in tp.get('B48', []):
            text = pdf.pages[page_idx].extract_text()
            dod_rows, civ_rows = parse_race_page(text)
            dod_race_rows.extend(dod_rows)
            civ_race_rows.extend(civ_rows)
    
        df_gender = pd.DataFrame(gender_rows).drop_duplicates(subset='state')
        df_gender['year'] = year
            
        df_dod_race = pd.DataFrame(dod_race_rows).drop_duplicates(subset='state')
        df_civ_race = pd.DataFrame(civ_race_rows).drop_duplicates(subset='state')
        df_dod_race['year'] = year
        df_civ_race['year'] = year
        print(f'DoD race rows: {len(df_dod_race)}')
        print(f'Civilian race rows: {len(df_civ_race)}')

    p4_years_gender.append(df_gender)
    p4_years_dod_race.append(df_dod_race)
    p4_years_civ_race.append(df_civ_race)

DoD race rows: 51
Civilian race rows: 51
DoD race rows: 51
Civilian race rows: 51
DoD race rows: 51
Civilian race rows: 51


In [49]:
#stack all years
df_gender_p4 = pd.concat(p4_years_gender, ignore_index=True)
df_dod_race_p4 = pd.concat(p4_years_dod_race, ignore_index=True)
df_civ_race_p4 = pd.concat(p4_years_civ_race, ignore_index=True)

In [51]:
# Check 1: correct row counts
print('=== Row counts ===')
print(f'Gender rows: {len(df_gender_p4)} (expected {51 * len(period4_files)})')
print(f'DoD race rows: {len(df_dod_race_p4)} (expected {51 * len(period4_files)})')
print(f'Civilian race rows: {len(df_civ_race_p4)} (expected {51 * len(period4_files)})')
print(f'Years covered: {sorted(df_gender_p4.year.unique())}')
print()

# Check 2: no duplicate state-year combinations
dupes_gender_p4 = df_gender_p4[df_gender_p4.duplicated(subset=['state','year'])]
dupes_race_p4 = df_dod_race_p4[df_dod_race_p4.duplicated(subset=['state','year'])]
print(f'Gender duplicates: {len(dupes_gender)}')
print(f'Race duplicates: {len(dupes_race)}')
print()

# Check 3: spot check known values from the PDFs
checks = [
    (2020, 'Tennessee', 3337),
    (2020, 'Florida',   12200),
    (2020, 'Texas',     17200),
    (2020, 'California', None),  # just print whatever we get
]

print('=== Spot checks ===')
for year, state, expected in checks:
    row = df_gender_p4[
        (df_gender_p4.year == year) & 
        (df_gender_p4.state == state)
    ]
    if len(row) == 0:
        print(f'{year} {state}: NOT FOUND')
        continue
    actual = row['dod_total'].values[0]
    if expected:
        status = 'OK' if actual == expected else f'MISMATCH (expected {expected})'
        print(f'{year} {state}: {actual} {status}')
    else:
        print(f'{year} {state}: {actual}')
print()

# Check 4: verify all 51 states present for each year
print('=== States per year ===')
for year in sorted(df_gender_p4.year.unique()):
    n = len(df_gender_p4[df_gender_p4.year == year])
    status = 'OK' if n == 51 else f'WARNING - only {n}'
    print(f'{year}: {n} states {status}')

=== Row counts ===
Gender rows: 153 (expected 153)
DoD race rows: 153 (expected 153)
Civilian race rows: 153 (expected 153)
Years covered: [2020, 2021, 2022]

Gender duplicates: 0
Race duplicates: 0

=== Spot checks ===
2020 Tennessee: 3337.0 OK
2020 Florida: 12200.0 OK
2020 Texas: 17200.0 OK
2020 California: 17763.0

=== States per year ===
2020: 51 states OK
2021: 51 states OK
2022: 51 states OK


In [52]:
print('=== States per year ===')
for year in sorted(df_gender_p4.year.unique()):
    n_gender = len(df_gender_p4[df_gender_p4.year == year])
    n_dod = len(df_dod_race_p4[df_dod_race_p4.year == year])
    n_civ = len(df_civ_race_p4[df_civ_race_p4.year == year])
    status = 'OK' if n_gender == 51 and n_dod == 51 and n_civ == 51 else 'WARNING'
    print(f'{year}: gender={n_gender}, dod_race={n_dod}, civ_race={n_civ} {status}')

=== States per year ===
2020: gender=51, dod_race=51, civ_race=51 OK
2021: gender=51, dod_race=51, civ_race=51 OK
2022: gender=51, dod_race=51, civ_race=51 OK


In [53]:
#merge gender, race
df_p4 = pd.merge(df_gender_p4, df_dod_race_p4, on=['state', 'year'], how='outer')
df_p4 = pd.merge(df_p4, df_civ_race_p4, on=['state', 'year'], how='outer')

In [54]:
#civilian count
df_p4['civ_males'] = df_p4['civ_total'] * df_p4['civ_males_pct']/100
df_p4['civ_females'] = df_p4['civ_total'] * df_p4['civ_females_pct']/100

In [55]:
print(f'Period 1 panel shape: {df_p4.shape}')
print(f'Columns: {list(df_p4.columns)}')

Period 1 panel shape: (153, 43)
Columns: ['state', 'dod_males', 'dod_males_pct', 'dod_females', 'dod_females_pct', 'dod_total', 'dod_total_pct_x', 'civ_males_pct', 'civ_females_pct', 'civ_total_pct_x', 'year', 'dod_white', 'dod_white_pct', 'dod_black', 'dod_black_pct', 'dod_aian', 'dod_aian_pct', 'dod_asian', 'dod_asian_pct', 'dod_nhpi', 'dod_nhpi_pct', 'dod_two_or_more', 'dod_two_pct', 'dod_unknown', 'dod_unknown_pct', 'dod_total_race', 'dod_total_pct_y', 'civ_white', 'civ_white_pct', 'civ_black', 'civ_black_pct', 'civ_aian', 'civ_aian_pct', 'civ_asian', 'civ_asian_pct', 'civ_nhpi', 'civ_nhpi_pct', 'civ_two_or_more', 'civ_two_pct', 'civ_total', 'civ_total_pct_y', 'civ_males', 'civ_females']


In [56]:
df_all_periods = pd.concat([df_2007_2008, df_p2, df_p3, df_p4], ignore_index=True)

In [57]:
print(f'Shape: {df_all_periods.shape}')
print(f'Expected rows: {51 * 16}')
print(f'Years: {sorted(df_all_periods.year.unique())}')
print(f'States: {df_all_periods.state.nunique()}')
print()

# Check for duplicates
dupes = df_all_periods[df_all_periods.duplicated(subset=['state','year'])]
print(f'Duplicates: {len(dupes)}')
print()

# Check missing values in key columns
key_cols = ['dod_total', 'dod_males', 'dod_females', 'civ_total']
print('Missing values:')
for col in key_cols:
    if col in df_all_periods.columns:
        print(f'  {col}: {df_all_periods[col].isna().sum()}')

Shape: (816, 43)
Expected rows: 816
Years: [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
States: 51

Duplicates: 0

Missing values:
  dod_total: 0
  dod_males: 0
  dod_females: 0
  civ_total: 0


In [58]:
df_all_periods.to_csv('accessions_2007_2022.csv', index=False)
print('Saved accessions_2007_2022.csv')

Saved accessions_2007_2022.csv
